Fluxo principal de Demonstração RAG

*Baixar dependencias*

In [ ]:
%pip install --quiet --upgrade "langchain[aws]"
%pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph
%pip install --quiet --upgrade python-dotenv
%pip install -qU langchain-core


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

LANGSMITH_TRACING=True
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
LANGSMITH_API_KEY=os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT=os.getenv("LANGSMITH_PROJECT")

In [9]:
# Ensure your AWS credentials are configured

from langchain.chat_models import init_chat_model
from langchain_aws import BedrockEmbeddings

converse_model = init_chat_model("amazon.nova-micro-v1:0", model_provider="bedrock_converse")
embeddings_model = BedrockEmbeddings(model_id="amazon.titan-embed-text-v2:0")

In [10]:
%pip install -qU langchain-core

Note: you may need to restart the kernel to use updated packages.


In [11]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings_model)

Agora começa o RAG

In [12]:
import bs4
from langchain import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [43]:
# Load and chunk contents of the blog
loader = WebBaseLoader(
    web_paths=("https://statusinvest.com.br/fundos-imobiliarios/bbig11",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header", "h1", "h2", "h3", "h4", "h5", "h6", "top-info d-flex flex-wrap justify-between mb-3 mb-md-5")
        )
    ),
)
docs = loader.load()

In [45]:
docs[0] = """Código de negociação
BTLG11
Administrador
BTG Pactual Serviços
Financeiros
Gestor
BTG Pactual Gestora de 
Recursos
Quantidade de Emissões
13
Periodicidade dos 
Rendimentos
Mensal
Prazo de Duração
Indeterminado
Taxa:
0,90% a.a. sobre o valor 
de Mercado do Fundo
Quantidade cotas
43.249.151
"""

In [46]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

# Index chunks
_ = vector_store.add_documents(documents=all_splits)

# Define prompt for question-answering
prompt = hub.pull("rlm/rag-prompt")

AttributeError: 'str' object has no attribute 'page_content'

In [19]:
# Define state for application
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str


# Define application steps
def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = converse_model.invoke(messages)
    return {"answer": response.content}

In [20]:
# Compile application and test
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [22]:
response = graph.invoke({"question": "quem e o administrador desse fundo?"})
print(response["answer"])

O administrador desse fundo é João Silva. Ele é responsável por gerenciar as operações e investimentos do fundo. João tem mais de 15 anos de experiência no setor financeiro.
